# Camera Discovery Live Test

Simplified 3-stage pipeline: `TargetResolver → CandidateDiscoveryEngine → ReviewAndValidationPipeline`.

LLMs are used as advisory evidence interpreters/rankers for target intent, geocoder candidate ranking, and candidate semantic review. Deterministic code/tools remain responsible for geometry verification, stream validation, trusted-output authorization, and final artifact writing.

This notebook clones the configured GitHub repository branch by default, installs it in editable mode with the optional Playwright extra, installs a headless Chromium browser for dynamic-page network capture, runs a live test, and then displays trusted or untrusted camera outputs. It does not patch source files from notebook cells.

Coordinate enrichment uses source coordinates first, then candidate metadata/title geocoding, then the optional LLM location-inference fallback. The LLM fallback only infers place-name query variants from stream URLs and metadata; Nominatim supplies coordinates, and verified target bounding boxes still decide whether inferred coordinates are accepted.

This notebook supports single-location and multi-location queries, for example:

```text
Get me all traffic cameras from California
Get me all cameras from Greenville, Texas
Get me all cameras from London, England and New York, New York
```

The query is intentionally user-controlled. Phrases such as `traffic cameras`, `weather cameras`, or `public live cameras` should be interpreted as camera-type intent, while place names such as `California`, `Greenville, Texas`, or `London, England` are target geography.


Dynamic camera pages are supported when a source row uses `type: dynamic`; the notebook installs Playwright so those rows can capture `.m3u8`, JSON feed, MapServer, FeatureServer, ArcGIS, and camera API network requests during real browser rendering.


The CLI run uses application progress events from `camera_discovery.cli run --progress-style events`. Notebook-only display helpers remain inside this notebook so progress bars update in place without terminal-frame spam or external notebook helper packages. The combined CLI log is still saved without patching repository source code.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil

# Colab/repo bootstrap settings. Override with env vars if needed.
REPO_URL = os.environ.get("CAMERA_DISCOVERY_REPO_URL", "https://github.com/dshipley71/camera-discovery.git")
REPO_BRANCH = os.environ.get("CAMERA_DISCOVERY_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("CAMERA_DISCOVERY_REPO_DIR", "/content/camera-discovery"))

print("Notebook bootstrap")
print("repo url:", REPO_URL)
print("branch:", REPO_BRANCH)
print("repo dir:", REPO_DIR)

# No source files are patched by this notebook. To test changes, push/update the GitHub branch
# selected above and rerun the clone/install cells.


In [ ]:
%cd /content

if REPO_DIR.exists():
    print(f"Removing existing repo directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]
print("$", " ".join(clone_cmd))
subprocess.run(clone_cmd, check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

src_path = REPO_DIR / "src"
assert (src_path / "camera_discovery").exists(), f"Missing package at {src_path / 'camera_discovery'}"

# Make package imports work immediately, even before editable install finishes.
os.environ["PYTHONPATH"] = str(src_path)
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Install the package with the optional Playwright extra. Playwright is not a
# default dependency of the package, but this notebook enables it so dynamic
# source rows can perform real browser/network capture for .m3u8 and feed URLs.
install_cmd = [sys.executable, "-m", "pip", "install", "-e", ".[playwright]", "--no-build-isolation"]
print("$", " ".join(install_cmd))
subprocess.run(install_cmd, check=True)

# Install Chromium for Playwright. In Colab this is required before dynamic-page
# capture can launch a headless browser. Set CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER=false
# only if the browser is already installed in your runtime.
INSTALL_PLAYWRIGHT_BROWSER = os.environ.get("CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER", "true").strip().lower() in {"1", "true", "yes", "on"}
if INSTALL_PLAYWRIGHT_BROWSER:
    browser_cmd = [sys.executable, "-m", "playwright", "install", "chromium"]
    print("$", " ".join(browser_cmd))
    subprocess.run(browser_cmd, check=True)
else:
    print("Skipping Playwright browser install because CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER=false")

# Verify Playwright imports before the live run. Dynamic capture remains optional:
# if no SOURCES.md row uses type: dynamic, the browser is not launched.
try:
    from playwright.sync_api import sync_playwright  # type: ignore
    print("Playwright import OK; dynamic source rows can use browser network capture.")
except Exception as exc:
    raise RuntimeError(f"Playwright import failed after installation: {exc!r}")


In [ ]:
import camera_discovery
print("camera_discovery import OK:", camera_discovery.__file__)

# Load provider secrets from Colab userdata when available.
# Configure these in Colab as needed:
#   OLLAMA_API_KEY
#   OPENAI_API_KEY
#   OPENAI_BASE_URL
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#   AWS_DEFAULT_REGION
try:
    from google.colab import userdata  # type: ignore
except Exception as exc:
    userdata = None
    print("Colab userdata not available:", repr(exc))

if userdata is not None:
    for key in [
        "OLLAMA_API_KEY",
        "OPENAI_API_KEY",
        "OPENAI_BASE_URL",
        "AWS_ACCESS_KEY_ID",
        "AWS_SECRET_ACCESS_KEY",
        "AWS_SESSION_TOKEN",
        "AWS_DEFAULT_REGION",
    ]:
        if os.environ.get(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
            print(f"Loaded {key} from Colab userdata")


# Sanity-check the installed CLI module without printing help/usage output.
import camera_discovery.cli as camera_cli
print("camera_discovery.cli import OK:", camera_cli.__file__)


| Profile    | Purpose                    | Behavior                                                                                                                                                                             |
| ---------- | -------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `fast`     | Quick review/discovery run | Validation is minimized/disabled; trusted `camera.geojson` should not be produced unless trust requirements are met; useful for `untrusted_camera_candidates.geojson` review output. |
| `balanced` | Middle-ground run          | More validation than Fast, but avoids the most expensive checks. Good default for routine testing.                                                                                   |
| `full`     | Most thorough run          | Runs the deepest validation path available, intended for trusted output when geometry and stream validation pass.                                                                    |


In [ ]:
# Notebook-only run/progress helpers.
# These helpers stay in the notebook because they are Colab/IPython display glue,
# not camera-discovery application source code.
from __future__ import annotations

# --- Notebook helper section: config.py content moved into the notebook ---

import os
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class NotebookRunSettings:
    """Settings needed by the notebook runner cell."""

    profile: str
    query: str
    output_dir: Path
    clean_output_dir: bool
    discovery_mode: str
    sources_file: Path
    seed_urls: list[str]
    llm_provider: str


def _bool_env(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().casefold() in {"1", "true", "yes", "on"}


def _split_csv(value: str) -> list[str]:
    return [part.strip() for part in value.split(",") if part.strip()]


def configure_notebook_run_from_env() -> NotebookRunSettings:
    """Read notebook run settings from environment and apply source-level defaults.

    The application configuration remains centralized in camera_discovery.core.config.
    This helper only validates notebook-facing settings and sets safe defaults for
    variables the notebook usually exposes to users.
    """

    profile = os.environ.get("CAMERA_DISCOVERY_PROFILE", "fast").strip().lower()
    if profile not in {"fast", "balanced", "full"}:
        raise ValueError(f"Invalid CAMERA_DISCOVERY_PROFILE={profile!r}; expected fast, balanced, or full")

    discovery_mode = os.environ.get("CAMERA_DISCOVERY_DISCOVERY_MODE", "both").strip().lower()
    if discovery_mode not in {"blind", "directory", "both", "direct"}:
        raise ValueError(f"Invalid CAMERA_DISCOVERY_DISCOVERY_MODE={discovery_mode!r}")

    os.environ.setdefault("CAMERA_DISCOVERY_LLM_PROVIDER", "ollama-cloud")
    os.environ.setdefault("CAMERA_DISCOVERY_LLM_MODEL", "gemma4:31b-cloud")
    os.environ.setdefault("CAMERA_DISCOVERY_TARGET_INTENT_MODEL", "gemma3:12b-cloud")
    os.environ.setdefault("CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL", "gemma3:12b-cloud")
    os.environ.setdefault("CAMERA_DISCOVERY_TARGET_INTENT_ATTEMPTS", "2")
    os.environ.setdefault("CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL", "gemma4:31b-cloud")
    os.environ.setdefault("CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL", "gemma4:31b-cloud")
    os.environ.setdefault("CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL", "gemma3:12b-cloud")
    os.environ.setdefault("CAMERA_DISCOVERY_ENABLE_LLM_LOCATION_INFERENCE", "true")
    os.environ.setdefault("CAMERA_DISCOVERY_LOCATION_INFERENCE_MIN_CONFIDENCE", "0.70")
    os.environ.setdefault("CAMERA_DISCOVERY_IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS", "2.0")

    query = os.environ.get("CAMERA_DISCOVERY_QUERY", "Get me all traffic cameras from California")
    output_dir = Path(os.environ.get("CAMERA_DISCOVERY_OUTPUT_DIR", "runs/notebook-live-test"))
    sources_file = Path(os.environ.get("CAMERA_DISCOVERY_SOURCES_FILE", "SOURCES.md"))
    seed_urls = _split_csv(os.environ.get("CAMERA_DISCOVERY_SEED_URLS", ""))
    if discovery_mode == "direct" and not seed_urls:
        raise ValueError("direct discovery mode requires CAMERA_DISCOVERY_SEED_URLS")

    return NotebookRunSettings(
        profile=profile,
        query=query,
        output_dir=output_dir,
        clean_output_dir=_bool_env("CAMERA_DISCOVERY_CLEAN_OUTPUT_DIR", True),
        discovery_mode=discovery_mode,
        sources_file=sources_file,
        seed_urls=seed_urls,
        llm_provider=os.environ["CAMERA_DISCOVERY_LLM_PROVIDER"],
    )


def print_notebook_run_settings(settings: NotebookRunSettings) -> None:
    """Print a concise, human-readable summary of notebook run settings."""

    secret_hint = {
        "ollama": "OLLAMA_API_KEY is required only when using Ollama Cloud; local Ollama may not need it.",
        "ollama-cloud": "OLLAMA_API_KEY is required.",
        "ollama_cloud": "OLLAMA_API_KEY is required.",
        "openai-compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
        "openai_compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
        "bedrock": "AWS credentials and AWS_DEFAULT_REGION are required.",
    }.get(settings.llm_provider.lower(), "provider-specific credentials are required")

    print("profile:", settings.profile)
    print("query:", settings.query)
    print("output:", settings.output_dir)
    print("clean output dir before run:", settings.clean_output_dir)
    print("provider:", settings.llm_provider)
    for key in [
        "CAMERA_DISCOVERY_LLM_MODEL",
        "CAMERA_DISCOVERY_TARGET_INTENT_MODEL",
        "CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL",
        "CAMERA_DISCOVERY_TARGET_INTENT_ATTEMPTS",
        "CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL",
        "CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL",
        "CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL",
        "CAMERA_DISCOVERY_MAX_HLS_CANDIDATES",
        "CAMERA_DISCOVERY_MAX_IMAGE_SNAPSHOT_CANDIDATES",
        "CAMERA_DISCOVERY_IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS",
    ]:
        if key in os.environ:
            print(f"{key.lower().replace('camera_discovery_', '').replace('_', ' ')}:", os.environ[key])
    print("discovery mode:", settings.discovery_mode)
    print("sources file:", settings.sources_file)
    print("seed urls:", len(settings.seed_urls))
    print("credential hint:", secret_hint)
    if settings.llm_provider.lower() in {"ollama-cloud", "ollama_cloud"} and not os.environ.get("OLLAMA_API_KEY"):
        print("WARNING: OLLAMA_API_KEY is not set; Ollama Cloud requests will fail until configured.")

# --- Notebook helper section: progress.py content moved into the notebook ---

import html
import json
from dataclasses import dataclass, field
from typing import Any

from camera_discovery.core.progress_events import PROGRESS_EVENT_PREFIX


@dataclass
class ProgressTaskState:
    """State for one notebook progress task."""

    key: str
    label: str
    completed: int = 0
    total: int | None = None
    detail: str = ""
    status: str = "active"

    @property
    def percent(self) -> float:
        if not self.total or self.total <= 0:
            return 0.0
        return max(0.0, min(100.0, (self.completed / self.total) * 100.0))

    def update(self, *, label: str | None = None, completed: int | None = None, total: int | None = None, detail: str | None = None, status: str | None = None) -> None:
        if label is not None:
            self.label = label
        if completed is not None:
            self.completed = max(0, int(completed))
        if total is not None:
            self.total = max(0, int(total))
        if detail is not None:
            self.detail = detail
        if status is not None:
            self.status = status


@dataclass
class NotebookProgressState:
    """Aggregate state rendered as a stable notebook progress panel."""

    title: str = "camera-discovery progress"
    root_tasks: dict[str, ProgressTaskState] = field(default_factory=dict)
    target_tasks: dict[str, dict[str, ProgressTaskState]] = field(default_factory=dict)
    target_labels: dict[str, str] = field(default_factory=dict)
    validation_counts: dict[str, int] = field(default_factory=dict)

    def root_task(self, key: str, label: str) -> ProgressTaskState:
        task = self.root_tasks.get(key)
        if task is None:
            task = ProgressTaskState(key=key, label=label)
            self.root_tasks[key] = task
        return task

    def target_task(self, target_id: str, task_key: str, label: str) -> ProgressTaskState:
        target_map = self.target_tasks.setdefault(target_id, {})
        task = target_map.get(task_key)
        if task is None:
            task = ProgressTaskState(key=task_key, label=label)
            target_map[task_key] = task
        return task

    def handle_event(self, event: str, payload: dict[str, Any]) -> None:
        target_id = str(payload.get("target_id") or "target")
        target_label = str(payload.get("target_label") or payload.get("label") or target_id)
        if target_label and target_label != "None":
            self.target_labels[target_id] = target_label

        if event == "target_resolution_started":
            total = int(payload.get("total") or 1)
            self.root_task("targets", "Resolving targets").update(completed=int(payload.get("completed") or 0), total=total, detail=f"{total} target(s)")
            return
        if event == "target_resolution_complete":
            total = int(payload.get("total") or payload.get("targets") or 1)
            completed = int(payload.get("completed") or payload.get("targets") or total)
            self.root_task("targets", "Resolving targets").update(completed=completed, total=max(total, completed, 1), detail=f"{completed} resolved", status="complete")
            return
        if event == "target_discovery_started":
            self.target_task(target_id, "scan", "Scanning source rows").update(total=None, completed=0, detail="loading")
            return
        if event == "source_rows_loading":
            self.target_task(target_id, "scan", "Scanning source rows").update(detail="loading source rows")
            return
        if event == "source_rows_selected":
            total = int(payload.get("primary_rows") or payload.get("selected_rows") or 0)
            selected = int(payload.get("selected_rows") or total)
            discovered = int(payload.get("discovered_rows") or selected)
            self.target_task(target_id, "scan", "Scanning source rows").update(total=total, completed=0, detail=f"{selected} selected · {discovered} discovered")
            return
        if event == "source_row_batch_started":
            rows = int(payload.get("rows") or 0)
            task = self.target_task(target_id, "scan", "Scanning source rows")
            if payload.get("phase") != "primary" and rows:
                task.update(total=(task.total or 0) + rows, detail=f"checking promoted rows · {rows}")
            elif rows and not task.total:
                task.update(total=rows)
            return
        if event == "source_row_processed":
            processed = int(payload.get("processed_rows") or 0)
            rows = int(payload.get("rows") or 0)
            total = rows or None
            detail = f"accepted {payload.get('accepted_total', 0)} · HLS {payload.get('hls_count', 0)} · images {payload.get('image_snapshot_count', 0)}"
            self.target_task(target_id, "scan", "Scanning source rows").update(total=total, completed=processed, detail=detail)
            return
        if event == "source_row_batch_complete":
            processed = int(payload.get("processed_rows") or payload.get("rows") or 0)
            rows = int(payload.get("rows") or processed)
            self.target_task(target_id, "scan", "Scanning source rows").update(total=max(rows, processed), completed=processed, status="complete")
            return
        if event == "coordinate_enrichment_started":
            total = int(payload.get("unique") or payload.get("total") or 0)
            already = int(payload.get("already_coordinate_bearing") or 0)
            self.target_task(target_id, "coordinates", "Enriching coordinates").update(total=total, completed=0, detail=f"mapped {already} · metadata 0 · geocoded 0 · LLM 0")
            return
        if event == "coordinate_candidate_processed":
            total = int(payload.get("total") or 0)
            completed = int(payload.get("processed") or 0)
            detail = (
                f"mapped {payload.get('coordinate_bearing', 0)} · "
                f"metadata {payload.get('metadata_enriched', 0)} · "
                f"geocoded {payload.get('geocode_enriched', 0)} · "
                f"LLM {payload.get('llm_location_enriched', 0)}"
            )
            self.target_task(target_id, "coordinates", "Enriching coordinates").update(total=total, completed=completed, detail=detail)
            return
        if event == "coordinate_enrichment_complete":
            total = int(payload.get("total") or 0)
            detail = (
                f"mapped {payload.get('coordinate_bearing', 0)} · "
                f"metadata {payload.get('metadata_enriched', 0)} · "
                f"geocoded {payload.get('geocode_enriched', 0)} · "
                f"LLM {payload.get('llm_location_enriched', 0)}"
            )
            self.target_task(target_id, "coordinates", "Enriching coordinates").update(total=total, completed=total, detail=detail, status="complete")
            return
        if event == "scope_review_started":
            total = int(payload.get("unique") or 1)
            self.target_task(target_id, "scope", "Scope and review gates").update(total=total, completed=0, detail="reviewing")
            return
        if event == "discovery_complete":
            total = int(payload.get("unique") or 1)
            detail = f"raw {payload.get('raw', 0)} · unique {payload.get('unique', 0)} · mapped {payload.get('coordinate_bearing', 0)}"
            self.target_task(target_id, "scope", "Scope and review gates").update(total=max(total, 1), completed=max(total, 1), detail=detail, status="complete")
            return
        if event == "validation_started":
            total = int(payload.get("total") or 1)
            self.root_task("validation", "Validation and outputs").update(total=total, completed=int(payload.get("completed") or 0), detail=str(payload.get("description") or "starting"))
            return
        if event.startswith("validation_") and event not in {"validation_started", "validation_complete"}:
            self._handle_validation_progress(event, payload)
            return
        if event == "validation_complete":
            total = int(payload.get("total") or 1)
            completed = int(payload.get("completed") or total)
            self.root_task("validation", "Validation and outputs").update(total=max(total, completed, 1), completed=completed, detail=str(payload.get("description") or "complete"), status="complete")
            return

    def _handle_validation_progress(self, event: str, payload: dict[str, Any]) -> None:
        total = int(payload.get("total") or 0)
        completed = int(payload.get("completed") or payload.get("processed") or 0)
        for key in ("live", "dead", "offline", "restricted", "decode_failed", "static_image_asset", "unknown"):
            if key in payload:
                self.validation_counts[key] = int(payload.get(key) or 0)
        detail_parts = []
        for key, label in (("live", "live"), ("dead", "dead"), ("offline", "offline"), ("restricted", "restricted"), ("decode_failed", "decode failed"), ("static_image_asset", "static asset"), ("unknown", "unknown")):
            if key in self.validation_counts:
                detail_parts.append(f"{label} {self.validation_counts[key]}")
        detail = " · ".join(detail_parts) if detail_parts else str(payload.get("description") or event.replace("_", " "))
        if event in {"validation_hls_started", "validation_hls_progress", "validation_hls_complete"}:
            label = "Validating HLS playlists"
            status = "complete" if event.endswith("complete") else "active"
            self.root_task("validation_hls", label).update(total=total, completed=completed, detail=detail, status=status)
        elif event in {"validation_image_started", "validation_image_progress", "validation_image_complete"}:
            label = "Validating image snapshots"
            status = "complete" if event.endswith("complete") else "active"
            self.root_task("validation_images", label).update(total=total, completed=completed, detail=detail, status=status)
        elif event in {"outputs_started", "outputs_progress", "outputs_complete", "packaging_started", "packaging_complete"}:
            label = "Writing outputs" if event.startswith("outputs") else "Packaging review artifacts"
            status = "complete" if event.endswith("complete") else "active"
            key = "outputs" if event.startswith("outputs") else "packaging"
            self.root_task(key, label).update(total=max(total, 1), completed=completed, detail=detail, status=status)


def parse_progress_event_line(line: str) -> tuple[str, dict[str, Any]] | None:
    """Parse a progress event line emitted by the CLI, if present."""

    if not line.startswith(PROGRESS_EVENT_PREFIX):
        return None
    raw = line[len(PROGRESS_EVENT_PREFIX) :].strip()
    if not raw:
        return None
    message = json.loads(raw)
    event = str(message.get("event") or "")
    payload = message.get("payload") or {}
    if not isinstance(payload, dict):
        payload = {}
    return event, payload


def _render_bar(task: ProgressTaskState) -> str:
    total = task.total
    if total and total > 0:
        label = f"{task.completed}/{total} · {task.percent:5.1f}%"
        width = task.percent
    else:
        label = f"{task.completed}"
        width = 0.0
    status = " ✓" if task.status == "complete" else ""
    detail = f"<div class='cd-detail'>{html.escape(task.detail)}</div>" if task.detail else ""
    return f"""
    <div class='cd-task'>
      <div class='cd-row'><span class='cd-label'>{html.escape(task.label)}{status}</span><span class='cd-count'>{html.escape(label)}</span></div>
      <div class='cd-bar'><div class='cd-fill' style='width:{width:.3f}%'></div></div>
      {detail}
    </div>
    """


def render_progress_html(state: NotebookProgressState) -> str:
    """Render notebook progress state as one stable HTML panel."""

    parts = [
        """
<style>
.camera-discovery-progress {font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; border:1px solid #d7dde8; border-radius:12px; padding:14px; margin:8px 0; background:#fff; color:#172033; max-width:980px;}
.camera-discovery-progress .cd-title {font-weight:700; margin-bottom:10px; font-size:15px;}
.camera-discovery-progress .cd-section {margin-top:12px; padding-top:8px; border-top:1px solid #eef1f6;}
.camera-discovery-progress .cd-target {font-weight:650; margin:4px 0 8px; color:#24364f;}
.camera-discovery-progress .cd-task {margin:8px 0;}
.camera-discovery-progress .cd-row {display:flex; justify-content:space-between; gap:12px; align-items:baseline; font-size:13px;}
.camera-discovery-progress .cd-label {font-weight:600;}
.camera-discovery-progress .cd-count {font-variant-numeric: tabular-nums; color:#596579; white-space:nowrap;}
.camera-discovery-progress .cd-detail {font-size:12px; color:#667085; margin-top:2px;}
.camera-discovery-progress .cd-bar {height:10px; border-radius:999px; overflow:hidden; background:#e7ebf2; margin-top:4px;}
.camera-discovery-progress .cd-fill {height:100%; background:#2f6fed; border-radius:999px; transition:width .18s ease;}
</style>
        """,
        f"<div class='camera-discovery-progress'><div class='cd-title'>{html.escape(state.title)}</div>",
    ]
    for task in state.root_tasks.values():
        if task.key.startswith("validation_") or task.key in {"outputs", "packaging"}:
            continue
        parts.append(_render_bar(task))
    for target_id, tasks in state.target_tasks.items():
        target_label = state.target_labels.get(target_id, target_id)
        parts.append(f"<div class='cd-section'><div class='cd-target'>{html.escape(target_label)}</div>")
        for key in ("scan", "coordinates", "scope"):
            task = tasks.get(key)
            if task is not None:
                parts.append(_render_bar(task))
        parts.append("</div>")
    validation_tasks = [state.root_tasks[k] for k in ("validation_hls", "validation_images", "outputs", "packaging", "validation") if k in state.root_tasks]
    if validation_tasks:
        parts.append("<div class='cd-section'><div class='cd-target'>Validation and outputs</div>")
        for task in validation_tasks:
            parts.append(_render_bar(task))
        parts.append("</div>")
    parts.append("</div>")
    return "\n".join(parts)


class NotebookProgressRenderer:
    """In-place HTML renderer for CLI progress events inside IPython notebooks."""

    def __init__(self, *, title: str = "camera-discovery progress"):
        self.state = NotebookProgressState(title=title)
        self._display_handle: Any | None = None
        self._html_class: Any | None = None
        self._display_fn: Any | None = None
        try:
            from IPython.display import HTML, display

            self._html_class = HTML
            self._display_fn = display
        except Exception:
            self._html_class = None
            self._display_fn = None

    def handle_event(self, event: str, payload: dict[str, Any]) -> None:
        self.state.handle_event(event, payload)
        self.refresh()

    def refresh(self) -> None:
        if self._html_class is None or self._display_fn is None:
            return
        html_obj = self._html_class(render_progress_html(self.state))
        if self._display_handle is None:
            self._display_handle = self._display_fn(html_obj, display_id=True)
        else:
            self._display_handle.update(html_obj)

# --- Notebook helper section: runner.py content moved into the notebook ---

import os
import shutil
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence



@dataclass(frozen=True)
class NotebookRunResult:
    """Result from running the camera-discovery CLI in a notebook."""

    returncode: int
    command: list[str]
    combined_log: Path
    progress_events_log: Path
    output_dir: Path


def build_camera_discovery_command(settings: NotebookRunSettings) -> list[str]:
    """Build the CLI command used by the live-test notebook."""

    cmd = [
        sys.executable,
        "-u",
        "-m",
        "camera_discovery.cli",
        "run",
        settings.query,
        "--profile",
        settings.profile,
        "--output-dir",
        str(settings.output_dir),
        "--discovery-mode",
        settings.discovery_mode,
        "--sources-file",
        str(settings.sources_file),
        "--progress",
        "--progress-style",
        "events",
    ]
    for url in settings.seed_urls:
        cmd.extend(["--seed-url", url])
    return cmd


def _safe_remove_output_dir(output_dir: Path, *, repo_dir: Path | None = None) -> None:
    resolved_output = output_dir.resolve()
    unsafe_roots = {Path("/").resolve(), Path("/content").resolve()}
    if repo_dir is not None:
        unsafe_roots.add(repo_dir.resolve())
    if resolved_output in unsafe_roots:
        raise RuntimeError(f"Refusing to remove unsafe output directory: {resolved_output}")
    if output_dir.exists():
        print(f"Removing stale output directory: {resolved_output}")
        shutil.rmtree(output_dir)


def run_camera_discovery_cli(
    settings: NotebookRunSettings,
    *,
    repo_dir: Path | None = None,
    src_path: Path | None = None,
    extra_env: dict[str, str] | None = None,
    renderer: NotebookProgressRenderer | None = None,
) -> NotebookRunResult:
    """Run the camera-discovery CLI with notebook-native progress rendering.

    The CLI emits machine-readable progress events. This runner captures those
    events for a stable in-place progress panel while still writing normal CLI
    output to a combined log file.
    """

    if settings.clean_output_dir:
        _safe_remove_output_dir(settings.output_dir, repo_dir=repo_dir)
    settings.output_dir.mkdir(parents=True, exist_ok=True)
    combined_log = settings.output_dir / "notebook_cli_combined.log"
    progress_events_log = settings.output_dir / "notebook_progress_events.jsonl"
    cmd = build_camera_discovery_command(settings)

    print("$", " ".join(cmd))
    print("Streaming CLI output live; notebook-native progress bars update in place without terminal spam.")
    print("combined log:", combined_log)
    print("progress events:", progress_events_log)

    run_env = os.environ.copy()
    if src_path is not None:
        run_env["PYTHONPATH"] = str(src_path)
    if extra_env:
        run_env.update(extra_env)

    progress_renderer = renderer or NotebookProgressRenderer()
    process = subprocess.Popen(
        cmd,
        stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=run_env,
    )
    assert process.stdout is not None
    with combined_log.open("w", encoding="utf-8") as log_fh, progress_events_log.open("w", encoding="utf-8") as event_fh:
        for line in process.stdout:
            parsed = parse_progress_event_line(line)
            if parsed is not None:
                event, payload = parsed
                event_fh.write(line)
                event_fh.flush()
                progress_renderer.handle_event(event, payload)
                continue
            log_fh.write(line)
            log_fh.flush()
            print(line, end="")
    returncode = process.wait()
    print("\nexit:", returncode)
    print("combined CLI log:", combined_log)
    if returncode != 0:
        raise RuntimeError("camera-discovery run failed; inspect notebook_cli_combined.log and run artifacts")
    return NotebookRunResult(
        returncode=returncode,
        command=cmd,
        combined_log=combined_log,
        progress_events_log=progress_events_log,
        output_dir=settings.output_dir,
    )

RUN_SETTINGS = configure_notebook_run_from_env()
RUN_PROFILE = RUN_SETTINGS.profile
USER_QUERY = RUN_SETTINGS.query
OUTPUT_DIR = RUN_SETTINGS.output_dir
CLEAN_OUTPUT_DIR = RUN_SETTINGS.clean_output_dir
DISCOVERY_MODE = RUN_SETTINGS.discovery_mode
SOURCES_FILE = RUN_SETTINGS.sources_file
SEED_URLS = RUN_SETTINGS.seed_urls

print_notebook_run_settings(RUN_SETTINGS)


In [ ]:
RUN_RESULT = run_camera_discovery_cli(
    RUN_SETTINGS,
    repo_dir=REPO_DIR,
    src_path=src_path,
)


In [ ]:
from pathlib import Path
import json

for rel in ["logs/source_policy_summary.json", "logs/candidate_discovery_summary.json", "logs/run_summary.json"]:
    path = OUTPUT_DIR / rel
    print("---", rel, "exists=", path.exists())
    if path.exists():
        try:
            print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)[:4000])
        except Exception as exc:
            print("Could not parse JSON:", repr(exc))
            print(path.read_text(encoding="utf-8")[:1000])


In [ ]:
summary_path = OUTPUT_DIR / "logs" / "run_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
targets = summary.get("targets", [])
print("targets:", len(targets))
for t in targets:
    print(json.dumps({
        "target_id": t.get("target_id"),
        "target_label": t.get("target_label"),
        "canonical_target": t.get("canonical_target"),
        "geometry_status": t.get("geometry_status"),
        "bbox_verified": t.get("bbox_verified"),
        "trust_policy": t.get("trust_policy"),
    }, indent=2))

candidate_summary = summary.get("candidates", {}) or {}
output_summary = summary.get("outputs", {}) or {}
unique_value = candidate_summary.get("unique_count")
if unique_value is None and isinstance(candidate_summary.get("unique"), list):
    unique_value = len(candidate_summary.get("unique"))
coord_value = candidate_summary.get("coordinate_bearing_count")
if coord_value is None and isinstance(candidate_summary.get("coordinate_bearing"), list):
    coord_value = len(candidate_summary.get("coordinate_bearing"))

print(json.dumps({
    "unique_candidates": unique_value,
    "coordinate_bearing": coord_value,
    "trusted_geojson_features": output_summary.get("trusted_geojson_features_written", 0),
    "untrusted_geojson_features": output_summary.get("untrusted_geojson_features_written", 0),
    "trusted_geojson_created": output_summary.get("trusted_geojson_created"),
    "untrusted_geojson_created": output_summary.get("untrusted_geojson_created"),
}, indent=2))


In [ ]:
from IPython.display import Markdown, display

for rel in [
    'camera.geojson',
    'untrusted_camera_candidates.geojson',
    'map.html',
    'notebook_camera_map.html',
    'review_artifacts.zip',
    'RUN_EXPLANATION.md',
    'logs/run_explanation.json',
    'logs/target_resolution_all.json',
    'logs/target_intent.json',
    'logs/geocoder_referee.json',
    'logs/candidate_semantic_review.json',
    'logs/candidate_coordinate_enrichment.json',
    'logs/structured_endpoint_discovery.jsonl',
    'logs/promoted_asset_host_rows.jsonl',
    'logs/playwright_network_capture_errors.jsonl',
    'logs/output_summary.json',
]:
    p = OUTPUT_DIR / rel
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

explanation_md = OUTPUT_DIR / 'RUN_EXPLANATION.md'
if explanation_md.exists():
    display(Markdown(explanation_md.read_text(encoding='utf-8')))
else:
    print('No RUN_EXPLANATION.md was written. Inspect logs/run_summary.json for raw state.')

# Per-target diagnostics are written below logs/targets/<target_id>/ and candidates/<target_id>/.
for folder in sorted((OUTPUT_DIR / 'logs' / 'targets').glob('*')) if (OUTPUT_DIR / 'logs' / 'targets').exists() else []:
    print('target diagnostics:', folder.relative_to(OUTPUT_DIR))


## Camera candidate table

This cell loads `camera_candidates_table.csv` first. That CSV is written from all non-rejected review candidates, including candidates that do **not** have latitude/longitude and therefore cannot be mapped. If the CSV is missing, the cell falls back to `camera.geojson` or `untrusted_camera_candidates.geojson`.

In [ ]:
from pathlib import Path
from IPython.display import display

TABLE_PATH = OUTPUT_DIR / "camera_candidates_table.csv"
CAMERA_ROWS = []
GEOJSON_PATH = None

if TABLE_PATH.exists() and TABLE_PATH.stat().st_size > 0:
    print("Selected table:", TABLE_PATH)
    try:
        import pandas as pd
        df = pd.read_csv(TABLE_PATH)
        print("Rows:", len(df))
        display_cols = [
            "name", "target_label", "location_text", "camera_type", "camera_id", "latitude", "longitude",
            "stream_url", "source_url", "thumbnail_url", "media_type", "trust_level",
            "validation_status", "scope_status", "discovery_method", "coordinate_source",
            "review_required",
        ]
        existing_cols = [col for col in display_cols if col in df.columns]
        display(df[existing_cols].head(200))
        CAMERA_ROWS = df.to_dict("records")
        if {"latitude", "longitude"}.issubset(df.columns):
            coordinate_rows = df[df["latitude"].notna() & df["longitude"].notna()]
            print("Coordinate-bearing table rows:", len(coordinate_rows))
        else:
            print("Coordinate-bearing table rows: 0 (latitude/longitude columns missing)")
    except Exception as exc:
        print("Could not display candidate CSV table:", repr(exc))
else:
    print("No camera_candidates_table.csv found; falling back to GeoJSON.")
    from camera_discovery.utils.geojson_viewer import (
        load_camera_rows,
        select_camera_geojson,
        write_camera_table_csv,
    )
    GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)
    print("Selected GeoJSON:", GEOJSON_PATH)
    if GEOJSON_PATH is None:
        CAMERA_ROWS = []
        print("No trusted or untrusted camera GeoJSON found yet.")
    else:
        CAMERA_ROWS = load_camera_rows(GEOJSON_PATH)
        table_csv = write_camera_table_csv(OUTPUT_DIR, CAMERA_ROWS)
        print("Rows:", len(CAMERA_ROWS))
        print("CSV table:", table_csv)
        if not CAMERA_ROWS:
            print("GeoJSON exists but contains no camera features.")
        else:
            try:
                import pandas as pd
                df = pd.DataFrame(CAMERA_ROWS)
                display(df.head(200))
            except Exception:
                for row in CAMERA_ROWS[:25]:
                    print(row)


## Interactive camera map

The map below uses the selected trusted or untrusted GeoJSON. Click a marker to see camera metadata, including camera type and camera ID when available. HLS candidates get a video player button. Image snapshot candidates get a refreshing image viewer instead of a broken video player.

In Colab, the notebook uses an `IFrame` plus a direct file link fallback because inline HTML rendering can be blocked by the notebook environment.


In [ ]:
from IPython.display import IFrame, HTML, display
from camera_discovery.utils.geojson_viewer import select_camera_geojson, write_embedded_camera_map

MAP_GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)
print("Selected GeoJSON for map:", MAP_GEOJSON_PATH)

if MAP_GEOJSON_PATH is None:
    print("No GeoJSON available for map display yet. The table above may still contain non-coordinate candidates.")
else:
    MAP_PATH = write_embedded_camera_map(OUTPUT_DIR, MAP_GEOJSON_PATH, output_name="notebook_camera_map.html")
    print("Notebook map:", MAP_PATH)
    print("Open manually if the iframe is blank:", MAP_PATH.resolve())
    display(IFrame(src=str(MAP_PATH), width="100%", height=720))
    display(HTML(f'<p><a href="{MAP_PATH}" target="_blank">Open camera map in a new tab</a></p>'))
